# Workshop — cell composition of Pool_ctr, four ways

Each group demultiplexed the same pool (10,308 cells, 7 colorectal lines) with different information:

| Group | Run id | What Vireo was told |
|---|---|---|
| 1 | `group1_guided7` | all 7 lines: RKO, HCT116, HT29, HCT8, CACO2, DLD1, LIM1215 |
| 2 | `group2_unguided` | nothing — find 7 donors, then match them to the database |
| 3 | `group3_no_hct8` | 6 lines, HCT8 left out |
| 4 | `group4_no_dld1` | 6 lines, DLD1 left out |

**DLD1 and HCT8 come from the same patient.**

1. Set `MY_RUN` below to your group's run id and run the notebook (**Run All**).
2. Section 1 draws the cell composition of **your** run.
3. Section 2 puts all four groups side by side for the discussion.

In [1]:
# Your group's run id — one of:
#   'group1_guided7'   'group2_unguided'   'group3_no_hct8'   'group4_no_dld1'
MY_RUN = 'group1_guided7'

DEMO_DIR = '/groups/nils/data/scrna_pipeline/demo/results/demux'

In [2]:
import os
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# group label -> (the group's live run id, pre-computed copy of the same run)
GROUPS = {
    'Group 1 · guided, 7 lines': ('group1_guided7',  'precomputed_guided7'),
    'Group 2 · unguided':        ('group2_unguided', 'precomputed_unguided'),
    'Group 3 · guided, no HCT8': ('group3_no_hct8',  'precomputed_no_hct8'),
    'Group 4 · guided, no DLD1': ('group4_no_dld1',  'precomputed_no_dld1'),
}
POOL_LINES = ['RKO', 'HCT116', 'HT29', 'CACO2', 'LIM1215', 'DLD1', 'HCT8']

# Same colours as notebook 03; the doublet bar is split to show the DLD1/HCT8 pair
SEGMENTS = {
    'Matched (guided or high confidence)': '#2d6a4f',
    'Matched (low confidence)': '#f4a261',
    'Vireo only (unmatched)':  '#6d6875',
    'Doublet: HCT8+DLD1':      '#9d0208',
    'Doublet: other pair':     '#e63946',
    'Unassigned':              '#adb5bd',
}
FONT = dict(family='Arial, sans-serif', size=13, color='#1b1b1b')


def load_run(run_id):
    path = os.path.join(DEMO_DIR, run_id, 'final_assignments.tsv')
    return pd.read_csv(path, sep='\t') if os.path.exists(path) else None


def composition(fa):
    """One row per cell: the bar it belongs to and the colour segment within that bar."""
    def classify(row):
        src, line = row['source'], row['cell_line']
        if src == 'guided_match' or (src == 'db_match' and row['match_confidence'] != 'low'):
            return line, 'Matched (guided or high confidence)'
        if src == 'db_match':
            return line, 'Matched (low confidence)'
        if src == 'vireo_only':
            return line, 'Vireo only (unmatched)'
        if src == 'doublet':
            pair = set(line.replace('doublet:', '').split('+'))
            return 'doublet', 'Doublet: HCT8+DLD1' if pair == {'HCT8', 'DLD1'} else 'Doublet: other pair'
        return 'unassigned', 'Unassigned'
    out = fa.apply(classify, axis=1, result_type='expand')
    out.columns = ['bar', 'segment']
    return out


def bar_order(comp):
    """Matched lines by size, then unmatched Vireo donors by size, then doublet, unassigned."""
    sizes = comp['bar'].value_counts()
    named = [b for b in sizes.index if b not in ('doublet', 'unassigned') and not b.startswith('vireo:')]
    vireo = [b for b in sizes.index if b.startswith('vireo:')]
    return named + vireo + [b for b in ('doublet', 'unassigned') if b in sizes.index]


def add_bars(fig, comp, order, row=None, col=None, legend_seen=None):
    counts = comp.groupby(['bar', 'segment']).size().unstack(fill_value=0).reindex(order, fill_value=0)
    for seg, color in SEGMENTS.items():
        if seg not in counts.columns or counts[seg].sum() == 0:
            continue
        show = legend_seen is None or seg not in legend_seen
        if legend_seen is not None:
            legend_seen.add(seg)
        trace = go.Bar(x=order, y=counts[seg], name=seg, marker_color=color, legendgroup=seg,
                       showlegend=show, legendrank=list(SEGMENTS).index(seg), hovertemplate=f'{seg}<br>%{{x}}: %{{y:,}} cells<extra></extra>')
        if row is None:
            fig.add_trace(trace)
        else:
            fig.add_trace(trace, row=row, col=col)
    totals = counts.sum(axis=1)
    trace = go.Scatter(x=order, y=totals, mode='text', text=[f'{v:,}' for v in totals],
                       textposition='top center', textfont=dict(size=11, color='#1b1b1b'),
                       showlegend=False, hoverinfo='skip')
    if row is None:
        fig.add_trace(trace)
    else:
        fig.add_trace(trace, row=row, col=col)
    return totals


def resolve(run_id):
    """The run itself if it has finished, else its pre-computed copy."""
    fa = load_run(run_id)
    if fa is not None:
        return run_id, fa, False
    for live, pre in GROUPS.values():
        if run_id == live and load_run(pre) is not None:
            return pre, load_run(pre), True
    return run_id, None, False

## 1. Your run's cell composition

One bar per cell line (or unmatched Vireo donor), sorted by number of cells. Colours show **how** the cells were
assigned. The number on top of each bar is its total.

In [3]:
shown_id, fa, fallback = resolve(MY_RUN)
if fa is None:
    print(f'No results for {MY_RUN!r}. Check the spelling, or wait for the run to finish.')
else:
    if fallback:
        print(f'{MY_RUN} has not finished yet — showing its pre-computed copy ({shown_id}) for now. '
              f'Re-run this cell once your log says Done.')
    comp = composition(fa)
    order = bar_order(comp)
    fig = go.Figure()
    totals = add_bars(fig, comp, order)
    fig.update_layout(barmode='stack', title=f'Cell composition — {shown_id}  ({len(fa):,} cells)',
                      xaxis_title='Cell line / donor', yaxis_title='Cell count',
                      yaxis=dict(range=[0, totals.max() * 1.12], gridcolor='#ececec'),
                      legend_title='Assignment', height=460, width=900,
                      plot_bgcolor='white', paper_bgcolor='white', font=FONT)
    fig.show()
    display(totals.rename('cells').to_frame())

group1_guided7 has not finished yet — showing its pre-computed copy (precomputed_guided7) for now. Re-run this cell once your log says Done.


,cells
bar,
RKO,4502
HCT116,2968
HT29,722
CACO2,80
DLD1,33
LIM1215,30
HCT8,3
doublet,701
unassigned,1269


## 2. All four groups side by side

Same scale and the same bar order in every panel, so bars can be compared directly. A panel uses the group's own
run when it has finished, otherwise its pre-computed copy (marked in the panel title).

In [4]:
panels = []
for label, (live, pre) in GROUPS.items():
    shown_id, fa, fallback = resolve(live)
    if fa is None:
        print(f'{label}: no results found — skipped')
        continue
    panels.append((label, shown_id, fallback, composition(fa)))

# Shared bar order: the 7 pool lines, any other matched line, all unmatched donors together, doublet, unassigned
def shared_bar(b):
    return 'unmatched donors' if b.startswith('vireo:') else b
for i, (label, shown_id, fallback, comp) in enumerate(panels):
    comp = comp.copy()
    comp['bar'] = comp['bar'].map(shared_bar)
    panels[i] = (label, shown_id, fallback, comp)
extra = sorted({b for *_, c in panels for b in c['bar']} - set(POOL_LINES) - {'unmatched donors', 'doublet', 'unassigned'})
ORDER = POOL_LINES + extra + ['unmatched donors', 'doublet', 'unassigned']

titles = [f'{label}<br><sup>{shown_id}{" (pre-computed)" if fallback else ""}</sup>'
          for label, shown_id, fallback, _ in panels]
fig = make_subplots(rows=2, cols=2, subplot_titles=titles, shared_yaxes=True,
                    vertical_spacing=0.2, horizontal_spacing=0.05)
seen, all_totals = set(), {}
for k, (label, shown_id, fallback, comp) in enumerate(panels):
    all_totals[label] = add_bars(fig, comp, ORDER, row=k // 2 + 1, col=k % 2 + 1, legend_seen=seen)
ymax = max(t.max() for t in all_totals.values()) * 1.15
fig.update_yaxes(range=[0, ymax], gridcolor='#ececec')
fig.update_yaxes(title_text='Cell count', col=1)
fig.update_xaxes(tickangle=40)
fig.update_layout(barmode='stack', height=820, width=1250, legend_title='Assignment',
                  plot_bgcolor='white', paper_bgcolor='white', font=FONT)
fig.show()

table = pd.DataFrame(all_totals).reindex(ORDER).fillna(0).astype(int)
table

,"Group 1 · guided, 7 lines",Group 2 · unguided,"Group 3 · guided, no HCT8","Group 4 · guided, no DLD1"
bar,,,,
RKO,4502,3476,4504,4499
HCT116,2968,2306,2969,2961
HT29,722,726,724,726
CACO2,80,0,80,80
LIM1215,30,0,30,30
DLD1,33,0,342,0
HCT8,3,0,0,340
unmatched donors,0,1803,0,0
doublet,701,1335,540,549


## Discussion

- Which run assigned the most cells to a single line? Does more assigned mean more correct?
- In Group 1's run, why are so few cells called DLD1 or HCT8, and what is the dark red part of the doublet bar?
- Where did the HCT8 cells go in Group 3's run, and the DLD1 cells in Group 4's?
- Group 2 told Vireo nothing about the pool. Which lines did it still find, and what are the unmatched donors?
- What does this mean for designing a pool, and for reporting which cell lines you sequenced?